QPE without QECC, purely on the physical level

In [1]:
from qiskit import __version__
print(__version__)

2.1.1


In [2]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.quantum_info import Statevector, state_fidelity, Pauli, DensityMatrix, partial_trace
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit_aer.library import SaveDensityMatrix
from qiskit import transpile 
import numpy as np
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.circuit.library import HGate, UnitaryGate, IGate
import matplotlib.pyplot as plt

In [3]:
theta = np.arctan(np.sqrt((np.sqrt(5) - 1) / 2))
amp_0 = np.cos(theta/2)
amp_1 = np.sin(theta/2)

# Infidelity to Error Rates

In [4]:
def inf_to_error(infidelity_list, num_qubits):
    error_rates_list = []
    dimension = 2**num_qubits
    for infidelity in infidelity_list:
        error_rates_list.append( infidelity * (dimension / (dimension - 1)) )
    
    return error_rates_list

In [5]:
one_qubit_gate_infidelity = [2.8e-5 * i for i in range(1,6)]
two_qubit_gate_infidelity = [8.3e-4 * i for i in range(1,6)]
idle_gate_infidelity = [1.2e-4 * i for i in range(1,6)]
spam0 = [6.7e-4 * i for i in range(1,6)] # P(1|0), measured 1 given that 0 was prepared
spam1 = [1.2e-3 * i for i in range(1,6)] # P(0|1)

In [7]:
one_qubit_gate_error_prob = inf_to_error(one_qubit_gate_infidelity, num_qubits=1)
two_qubit_gate_error_prob = inf_to_error(two_qubit_gate_infidelity, num_qubits=2)
idle_gate_error_prob = inf_to_error(idle_gate_infidelity, num_qubits=1)

# Functions

In [8]:
def state_prep(qc: QuantumCircuit, alpha0: float, alpha1: float):
    qc.h(0)
    qc.x(1)
    qc.ry(alpha0, 1)
    qc.rz(alpha1, 1)

In [9]:
def control_u(qc: QuantumCircuit, h1: float, h2: float, t: float):
    qc.rz(h1*t, 1)
    qc.cx(0,1)
    qc.rz(-h1*t, 1)
    qc.cx(0,1)
    qc.h(1)
    qc.rz(h2*t, 1)
    qc.cx(0,1)
    qc.rz(-h2*t, 1)
    qc.cx(0,1)
    qc.h(1)

In [10]:
def qpe(qc: QuantumCircuit, alpha0: float, alpha1: float, h1: float, h2: float, t: float, beta: float, meas_bit: ClassicalRegister):
    state_prep(qc, alpha0, alpha1)
    control_u(qc, h1, h2, t)
    
    qc.rz(beta, 0)
    qc.h(0)
    qc.measure(0, meas_bit)

In [13]:
h1, h2, h3 = (0.79605, -0.18092, -0.32096)
print(h1)
print(h2)
print(h3)

0.79605
-0.18092
-0.32096
